In [ ]:
import pandas as pd

file_path = "../0.data/criteo-uplift-v2.1.csv"

df = pd.read_csv(file_path)

print(df.shape)
print(df.columns.tolist())
df.head()

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
print(df.columns.tolist())

['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']


In [ ]:
df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [ ]:
print(df.dtypes)

print(df[["treatment", "conversion", "visit", "exposure"]].value_counts())

print(df[["treatment", "conversion", "visit", "exposure"]].mean())

print(df.isna().sum())

f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object
treatment  conversion  visit  exposure
1          0           0      0           11055129
0          0           0      0            2016832
1          0           1      0             385634
                       0      1             250702
                       1      1             154479
0          0           1      0              76042
1          1           1      1              23031
                              0              13680
0          1           1      0               4063
Name: count, dtype: int64
treatment     0.850000
conversion    0.002917
visit         0.046992
exposure      0.030631
dtype: float64
f0 

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [f"f{i}" for i in range(12)]

sample_df = df.sample(
    n=1_000_000,
    random_state=42
)

X = sample_df[feature_cols]
W = sample_df["treatment"]
Y = sample_df["conversion"]

X_train, X_test, W_train, W_test, Y_train, Y_test = train_test_split(
    X,
    W,
    Y,
    test_size=0.3,
    random_state=42,
    stratify=W
)

In [ ]:
pip install econml scikit-learn pandas numpy matplotlib CausalForestDML

ERROR: Could not find a version that satisfies the requirement CausalForestDML (from versions: none)
ERROR: No matching distribution found for CausalForestDML
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from econml.dml import CausalForestDML

TypeError: C variable sklearn.utils._random.DEFAULT_SEED has wrong signature (expected __pyx_t_7sklearn_5utils_9_typedefs_uint32_t, got __pyx_t_7sklearn_5utils_9_typedefs_uint32_t const )

In [ ]:
feature_cols = [f"f{i}" for i in range(12)]

sample_df = df.sample(
    n=500_000,
    random_state=42
).copy()

# 処置×アウトカムの構成比を保つ
sample_df["strata"] = (
    sample_df["treatment"].astype(str)
    + "_"
    + sample_df["conversion"].astype(str)
)

train_df, test_df = train_test_split(
    sample_df,
    test_size=0.30,
    random_state=42,
    stratify=sample_df["strata"]
)

X_train = train_df[feature_cols].to_numpy()
W_train = train_df["treatment"].to_numpy()
Y_train = train_df["conversion"].to_numpy()

X_test = test_df[feature_cols].to_numpy()
W_test = test_df["treatment"].to_numpy()
Y_test = test_df["conversion"].to_numpy()

In [ ]:
cf = CausalForestDML(
    discrete_treatment=True,
    discrete_outcome=True,

    n_estimators=400,
    min_samples_leaf=100,
    max_depth=None,
    max_samples=0.45,

    honest=True,
    inference=True,

    cv=3,
    n_jobs=-1,
    random_state=42
)

cf.fit(
    Y_train,
    W_train,
    X=X_train
)

In [ ]:
cate_hat = cf.effect(X_test)

test_result = test_df[
    feature_cols + ["treatment", "conversion", "visit", "exposure"]
].copy()

test_result["cate_hat"] = cate_hat

test_result["cate_hat"].describe()

In [ ]:
test_result["cate_group"] = pd.qcut(
    test_result["cate_hat"],
    q=5,
    labels=["Q1_low", "Q2", "Q3", "Q4", "Q5_high"],
    duplicates="drop"
)

group_effect = (
    test_result
    .groupby(["cate_group", "treatment"], observed=True)["conversion"]
    .agg(["mean", "count"])
    .reset_index()
)

group_means = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="mean"
)

group_counts = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="count"
)

group_means["observed_itt"] = (
    group_means[1] - group_means[0]
)

print(group_means)
print(group_counts)